# Pharmacophore Overlay with PharmacophoresMT

A pharmacophore model encodes the spatial arrangement of essential interaction sites
(hydrogen-bond donors/acceptors, hydrophobic patches, charged groups, aromatic rings)
required for biological activity.  The **pharmacophoremt** addon renders these sites
as colour-coded glyphs directly on top of the protein structure in MolSysViewer.

This tutorial is a direct continuation of `tutorial_topomt_pocket.ipynb`: the same
CDK2 structure (PDB 1FIN) is used, and the pharmacophore is built inside the largest
pocket identified there.

**Requirements:** `pharmacophoremt`, `topomt`, `molsysmt`, `molsysviewer`,
`molsysviewer-pharmacophoremt`, `molsysviewer-topomt`.

In [ ]:
import molsysmt as msm
import topomt
import pharmacophoremt
from pharmacophoremt.modeler.structure_based import StructureBasedModeler
import molsysviewer as msv
from molsysviewer_topomt import get_addon as get_topomt_addon, lifecycle as topomt_lifecycle
from molsysviewer_topomt import build_view_with_topography
from molsysviewer_pharmacophoremt import (
    get_addon as get_pharma_addon,
    lifecycle as pharma_lifecycle,
    on_enable as pharma_on_enable,
    render_pharmacophore_elements,
)
from molsysviewer_pharmacophoremt.runtime import ensure_runtime as pharma_ensure_runtime

## Register both addons

TopoMT provides the structural pocket; PharmacophoresMT adds the interaction-site layer.

In [ ]:
msv.addons.register(get_topomt_addon(), lifecycle=topomt_lifecycle)
msv.addons.register(get_pharma_addon(), lifecycle=pharma_lifecycle)

## Load the protein and compute the topography

We repeat the setup from `tutorial_topomt_pocket.ipynb`.
If you have already run that notebook in the same session you can reuse `ms` and
`topography` directly.

In [ ]:
ms = msm.convert(
    "pdb_id:1fin",
    to_form="molsysmt.MolSys",
    selection='molecule_type == "protein"',
)

topography = topomt.get_topography(ms, method="pocketeer", structure_indices=0)
largest_pocket = sorted(topography.features, key=lambda f: f.volume, reverse=True)[0]
print(f"Target pocket: id={largest_pocket.id}  volume={largest_pocket.volume:.1f} Å³")

## Build the structure-based pharmacophore

`StructureBasedModeler` projects ideal hydrogen-bond vectors, hydrophobic centroids,
and charged/aromatic interaction points from the receptor atoms lining the pocket.

- `pocket_center` and `pocket_radius` constrain which receptor atoms are considered.
- `add_excluded_volumes=True` adds exclusion spheres for non-hydrogen pocket atoms,
  which is useful for 3D screening.

In [ ]:
pocket_center = largest_pocket.center   # (x, y, z) in Å
pocket_radius = largest_pocket.radius   # enclosing sphere radius in Å

modeler = StructureBasedModeler(
    molecular_system=ms,
    selection='molecule_type == "protein"',
    pocket_center=pocket_center,
    pocket_radius=pocket_radius,
)
pharmacophore = modeler.build(structure_indices=0, add_excluded_volumes=True)

n_sites = len(pharmacophore.interaction_sites)
print(f"Interaction sites: {n_sites}")

## Create the view with the pocket blob

We reuse `build_view_with_topography()` to get the protein + pocket surface,
then layer the pharmacophore glyphs on top.

In [ ]:
view = build_view_with_topography(
    ms,
    topography,
    feature_ids=[largest_pocket.id],
    selection='molecule_type == "protein"',
)
view.show()

## Attach the pharmacophore and render the glyphs

Each interaction-site type is rendered with a standard colour convention:

| Site type | Colour |
|-----------|--------|
| HBD (donor) | blue |
| HBA (acceptor) | red |
| HYD (hydrophobic) | yellow |
| PI (aromatic) | orange |
| CAT/ANI (charged) | green / magenta |
| ExVol (excluded vol.) | grey |

In [ ]:
# Enable the pharmacophoremt addon on this view
pharma_on_enable(view)

# Attach the pharmacophore to the runtime so the panel can access it
pharma_runtime = pharma_ensure_runtime(view)
pharma_runtime.pharmacophore = pharmacophore

# Render the interaction-site glyphs
render_pharmacophore_elements(view, pharmacophore, tag_prefix="pharma-cdk2")

## Toggle glyphs via the Pharmacophore panel

Once the runtime has a pharmacophore attached, the panel widget can clear and
re-render glyphs on demand — mirroring the interactive UI in the viewer sidebar.

In [ ]:
pharma_panel = view.addons.resolve_panel_widget("pharmacophoremt", "pharmacophore")

# Clear glyphs
pharma_panel.handle_action(view, "clear_pharmacophore", {})

# Re-render
pharma_panel.handle_action(view, "render_pharmacophore", {})

## Export the combined scene

In [ ]:
view.export.html(
    "1fin_pharmacophore.html",
    title="CDK2 — structure-based pharmacophore",
)

## Next steps

- Disable `add_excluded_volumes=False` to get a cleaner visual for presentation.
- Use `pharmacophoremt.screening` to screen a ligand library against this model.
- Combine with `tutorial_molsysmt_protein_inspection.ipynb` to inspect the protein
  residues lining the pocket before building the pharmacophore.